In [1]:
import requests
import json
import pandas as pd

# ================================
# Settings (Change to your own information)
# ================================
APP_KEY = "PSCMNK30YDpgTKOgD4tCOfVl9SEv0e73TCSv"
APP_SECRET = "uji809UwcuUbXnf3dH8623rJXXG0Tiql"

# LS증권 User ID
USER_ID = "eungizoa"
BASE_URL = "https://openapi.ls-sec.co.kr:8080"

def get_access_token(app_key: str, app_secret: str) -> str:
    """Issue an access token.

    Args:
        app_key: The app key.
        app_secret: The app secret.

    Returns:
        str: The issued access token string.
    """
    url = f"{BASE_URL}/oauth2/token"
    headers = {"content-type": "application/x-www-form-urlencoded"}
    body = {
        "grant_type": "client_credentials",
        "appkey": app_key,
        "appsecretkey": app_secret,
        "scope": "oob",
    }

    response = requests.post(url, headers=headers, data=body)
    response.raise_for_status()
    return response.json()["access_token"]

def get_server_condition_list(
    access_token: str,
    user_id: str,
    gb: str = "0",
    group_name: str = "",
    cont: str = "",
    cont_key: str = "",
) -> dict:
    """Query the list of server saved conditions (서버저장조건 리스트 조회, t1866).

    Args:
        access_token: Issued access token.
        user_id: Login ID (maximum 8 characters).
        gb: Query type.
            "0": Group + condition list.
            "1": Group list only.
            "2": Condition list for the given group name.
        group_name: Group name (used only if gb is "2").
        cont: Continuation flag (first query: "", for continued query: "1").
        cont_key: Continuation key (when performing continued query, use previous response's cont_key value).

    Returns:
        dict: Dictionary containing query results, format:
            {
                "t1866OutBlock":  { result_count, cont, cont_key },
                "t1866OutBlock1": [ { query_index, group_name, query_name }, ... ]
            }
    """
    url = f"{BASE_URL}/stock/item-search"

    headers = {
        "content-type": "application/json; charset=utf-8",
        "authorization": f"Bearer {access_token}",
        "tr_cd": "t1866",
        "tr_cont": "N",
        "tr_cont_key": cont_key,
    }

    body = {
        "t1866InBlock": {
            "user_id": user_id,
            "gb": gb,
            "group_name": group_name,
            "cont": cont,
            "cont_key": cont_key,
        }
    }

    response = requests.post(url, headers=headers, data=json.dumps(body))
    response.raise_for_status()

    result = response.json()

    # Check for API error
    if result.get("rsp_cd") != "00000":
        raise Exception(f"API error: [{result.get('rsp_cd')}] {result.get('rsp_msg')}")

    return result


def get_all_conditions(access_token: str, user_id: str, gb: str = "0", group_name: str = "") -> list:
    """Return the full list of server saved conditions by handling continued queries.

    Args:
        access_token: Access token.
        user_id: User ID.
        gb: Query type ("0", "1", "2").
        group_name: Group name if gb is "2".

    Returns:
        list: List of all condition items (each item contains query_index, group_name, query_name).
    """
    all_items = []
    cont = ""
    cont_key = ""

    while True:
        result = get_server_condition_list(
            access_token=access_token,
            user_id=user_id,
            gb=gb,
            group_name=group_name,
            cont=cont,
            cont_key=cont_key,
        )

        items = result.get("t1866OutBlock1", [])
        all_items.extend(items)

        out_block = result.get("t1866OutBlock", {})
        cont = out_block.get("cont", "")
        cont_key = out_block.get("cont_key", "") or out_block.get("contkey", "")

        # Finish if there is no more continued data
        if cont != "1":
            break

    return all_items

def search_by_condition(access_token: str, query_index: str) -> dict:
    """
    Search by saved server condition (t1859).

    Args:
        access_token: Access token issued for authentication.
        query_index: The query_index field value obtained from t1866, e.g. "testID0000".

    Returns:
        dict: 
            {
                "t1859OutBlock":  { result_count, result_time, text },
                "t1859OutBlock1": [ { shcode, hname, price, sign, change, diff, volume }, ... ]
            }

    Raises:
        Exception: If the API response code (rsp_cd) is not "00000".
    """
    url = f"{BASE_URL}/stock/item-search"
    headers = {
        "content-type": "application/json; charset=utf-8",
        "authorization": f"Bearer {access_token}",
        "tr_cd": "t1859",
        "tr_cont": "N",
        "tr_cont_key": "",
    }
    body = {
        "t1859InBlock": {
            "query_index": query_index,
        }
    }
    response = requests.post(url, headers=headers, data=json.dumps(body))
    response.raise_for_status()
    result = response.json()
    if result.get("rsp_cd") != "00000":
        raise Exception(f"API 오류: [{result.get('rsp_cd')}] {result.get('rsp_msg')}")
    return result

# 1. Issue access token
token = get_access_token(APP_KEY, APP_SECRET)

# 2. Query server condition list (all groups and conditions)
conditions = get_all_conditions(token, USER_ID, gb="0")

# 3. Print results
print(f"\nTotal {len(conditions)} screening conditions found.")
print("-" * 60)
print(f"{'Index':<15} {'Group Name':<20} {'Condition Name'}")
print("-" * 60)
for cond in conditions:
    print(f"{cond['query_index']:<15} {cond['group_name']:<20} {cond['query_name']}")


Total 1 screening conditions found.
------------------------------------------------------------
Index           Group Name           Condition Name
------------------------------------------------------------
eungizoa0000    BIC                  BIC


In [7]:
selected = None
for condition in conditions:
    if condition["query_name"] == "BIC":
        selected = condition
        break
else:
    raise ValueError(f"`BIC` condition not foundable in condition list: {[c['query_name'] for c in conditions]}")

print(f"Searching with screening query group `{selected['query_name']}`...")
result = search_by_condition(token, selected["query_index"])
out_block = result.get("t1859OutBlock", {})
stocks = result.get("t1859OutBlock1", [])

result_time = out_block.get("result_time", "")
formatted_time = f"{result_time[:2]}:{result_time[2:4]}:{result_time[4:]}" if len(result_time) == 6 else result_time
text = out_block.get("text", "")

print(f"Screened time: {formatted_time}")
print(f"Number of stocks screened: {out_block.get('result_count', 0)}")
if text:
    print(f"Screening explanation: {text}")

pd.DataFrame(stocks)

Searching with screening query group `BIC`...
Screened time: 14:50:08
Number of stocks screened: 6


,shcode,hname,price,sign,change,diff,volume
0,016090,대현,1807,2,10,0.56,109671
1,072020,중앙백신,10050,2,160,1.62,53488
2,086710,선진뷰티사이언스,10780,2,610,6.00,256339
3,136540,윈스테크넷,12750,2,50,0.39,45557
4,277880,티에스아이,5890,5,70,-1.17,167508
5,357230,에이치피오,2765,5,5,-0.18,231992
